# **4. Atualizar dados do site**

Este notebook fecha o pipeline: lê as saídas já processadas pelos notebooks anteriores (`0` a `3`, em especial o tagueamento do notebook `2`) e regenera os quatro blocos de dados embutidos no `index.html` deste repositório (`match-data`, `site-data`, `player-data` e `var-data`).

O restante do `index.html` (HTML, CSS, JavaScript de renderização) não é tocado — só os quatro blocos `<script type="application/json">` são substituídos.

Fluxo de atualização: rode `0` → `1` → `2` → `3` para atualizar os dados processados, depois rode este notebook para atualizar a página. Revise o diff do `index.html` e faça o commit/push deste repositório quando estiver satisfeito.

In [1]:
from __future__ import annotations

import json
import re
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

PAGE_HTML = ROOT / "index.html"
assert PAGE_HTML.exists(), f"não encontrei {PAGE_HTML} na raiz do repositório"

CARD_EVENTS_PATH = ROOT / "data" / "processed" / "keyword_tags" / "card_events_tagged.csv"
RUN_MANIFEST_PATH = ROOT / "data" / "processed" / "keyword_tags" / "run_manifest.json"
PLAYER_DATA_PATH = ROOT / "data" / "processed" / "keyword_tags" / "site_player_data.json"
GAMES_PATH = ROOT / "data" / "interim" / "games.csv"
VAR_MATCHES_PATH = ROOT / "data" / "processed" / "var_usage" / "match_var_usage.csv"
VAR_SEASON_PATH = ROOT / "data" / "processed" / "var_usage" / "var_profile_by_season.csv"
VAR_TEAM_PATH = ROOT / "data" / "processed" / "var_usage" / "var_profile_by_team.csv"
VAR_REFEREE_PATH = ROOT / "data" / "processed" / "var_usage" / "var_profile_by_referee.csv"
VAR_MANIFEST_PATH = ROOT / "data" / "processed" / "var_usage" / "run_manifest.json"

## **1. Normalização de nomes de time**

Os dados brutos (`games.csv`, `card_events_tagged.csv`) trazem variações do mesmo clube (`"Atlético Mineiro"`, `"Atlético Mineiro - MG"`, `"Coritiba SAF"`...). O site usa nomes curtos e únicos por clube (`"Atlético-MG"`, `"Coritiba"`...). Esta tabela faz essa canonicalização; foi construída comparando os 37 nomes brutos encontrados em `games.csv` com os 30 nomes exibidos hoje no site.

In [2]:
TEAM_ALIASES = {
    "america": "América-MG",
    "américa": "América-MG",
    "athletico paranaense": "Athletico-PR",
    "atlético": "Atlético-GO",
    "atlético goianiense saf": "Atlético-GO",
    "atlético mineiro": "Atlético-MG",
    "avaí": "Avaí",
    "bahia": "Bahia",
    "esporte clube bahia": "Bahia",
    "botafogo": "Botafogo",
    "botafogo de futebol e regatas": "Botafogo",
    "ceará": "Ceará",
    "chapecoense": "Chapecoense",
    "corinthians": "Corinthians",
    "coritiba": "Coritiba",
    "coritiba saf": "Coritiba",
    "criciúma": "Criciúma",
    "cruzeiro": "Cruzeiro",
    "cuiabá": "Cuiabá",
    "flamengo": "Flamengo",
    "fluminense": "Fluminense",
    "fortaleza esporte clube": "Fortaleza",
    "fortaleza saf": "Fortaleza",
    "goiás": "Goiás",
    "grêmio": "Grêmio",
    "internacional": "Internacional",
    "juventude": "Juventude",
    "mirassol": "Mirassol",
    "palmeiras": "Palmeiras",
    "red bull bragantino": "Red Bull Bragantino",
    "remo": "Remo",
    "santos fc": "Santos",
    "sport recife": "Sport",
    "são paulo": "São Paulo",
    "vasco da gama": "Vasco",
    "vasco da gama saf": "Vasco",
    "vitória": "Vitória",
}

TEAM_COLORS = {
    "Athletico-PR": ("#ff0302", "#000000"),
    "Atlético-GO": ("#c8102e", "#000000"),
    "Atlético-MG": ("#000000", "#ffffff"),
    "Avaí": ("#00457c", "#ffffff"),
    "Bahia": ("#0e3990", "#c8102e"),
    "Botafogo": ("#000000", "#ffffff"),
    "Ceará": ("#000000", "#ffffff"),
    "Chapecoense": ("#26913f", "#ffffff"),
    "Corinthians": ("#000000", "#ffffff"),
    "Coritiba": ("#2f6657", "#ffffff"),
    "Criciúma": ("#f4c300", "#000000"),
    "Cruzeiro": ("#013992", "#ffffff"),
    "Cuiabá": ("#0a6e33", "#f5b700"),
    "Flamengo": ("#bd2418", "#000000"),
    "Fluminense": ("#9d042d", "#046a38"),
    "Fortaleza": ("#1c3f94", "#e2001a"),
    "Goiás": ("#0d8a3e", "#ffffff"),
    "Grêmio": ("#1a86bf", "#000000"),
    "Internacional": ("#e40911", "#ffffff"),
    "Juventude": ("#1c8a43", "#ffffff"),
    "Mirassol": ("#f6ea17", "#0b6e2f"),
    "Palmeiras": ("#0b683e", "#ffffff"),
    "Red Bull Bragantino": ("#d72742", "#ffffff"),
    "Remo": ("#2a375e", "#c8102e"),
    "Santos": ("#000000", "#ffffff"),
    "São Paulo": ("#ed1d23", "#000000"),
    "Sport": ("#b0182f", "#000000"),
    "Vasco": ("#000000", "#ffffff"),
    "Vitória": ("#ff1505", "#000000"),
    "América-MG": ("#0f8140", "#ffffff"),
}

STATE_SUFFIX = re.compile(r" - [A-Z]{2}$")


def canon_team(raw: object) -> str:
    """Nome bruto do clube -> nome curto usado no site. Times não mapeados levantam KeyError
    de propósito: times novos na Série A precisam ser adicionados a TEAM_ALIASES antes de publicar."""
    if not isinstance(raw, str) or not raw.strip():
        return ""
    base = STATE_SUFFIX.sub("", raw).strip()
    key = base.lower()
    if key not in TEAM_ALIASES:
        raise KeyError(f"time não mapeado em TEAM_ALIASES: {raw!r} (adicione antes de publicar)")
    return TEAM_ALIASES[key]

## **2. Carregar dados processados**

In [3]:
df = pd.read_csv(CARD_EVENTS_PATH, low_memory=False)
manifest = json.loads(RUN_MANIFEST_PATH.read_text(encoding="utf-8"))
player_data = json.loads(PLAYER_DATA_PATH.read_text(encoding="utf-8"))
games_df = pd.read_csv(GAMES_PATH)
var_matches_df = pd.read_csv(VAR_MATCHES_PATH)
var_season_df = pd.read_csv(VAR_SEASON_PATH)
var_team_df = pd.read_csv(VAR_TEAM_PATH)
var_referee_df = pd.read_csv(VAR_REFEREE_PATH)
var_manifest = json.loads(VAR_MANIFEST_PATH.read_text(encoding="utf-8"))

TAG_ORDER = manifest["tags"]
TAG_LABEL = {t: manifest["taxonomy"][t]["label"] for t in TAG_ORDER}
CARD_CODE = {"yellow": "A", "second_yellow": "V2", "direct_red": "VD"}
PERIOD_LABEL = {"1T": "1º tempo", "2T": "2º tempo"}

len(df), len(player_data), len(games_df), len(var_matches_df)

(14146, 2215, 2660, 2570)

## **3. `match-data`: eventos por partida**

Um registro por jogo com todos os cartões daquela súmula. O rótulo de tempo de jogo replica o texto que aparece na súmula oficial: `"1º tempo · 18'"` para tempo normal, `"1º tempo · 45+3'"` para acréscimos (`clock` começando com `+`), e `"Intervalo"` / `"Após o jogo"` para os períodos `INT` / `PJ`.

In [4]:
def minute_label(period: str, minute: float, clock: object) -> str:
    if period == "INT":
        return "Intervalo"
    if period == "PJ":
        return "Após o jogo"
    base = PERIOD_LABEL.get(period, period)
    m = int(minute) if pd.notna(minute) else 0
    if isinstance(clock, str) and clock.startswith("+"):
        return f"{base} · 45+{m}\'"
    return f"{base} · {m}\'"


PERIOD_RANK = {"1T": 0, "INT": 1, "2T": 2, "PJ": 3}


def sort_minute(period: str, minute: float, clock: object) -> float:
    if period in ("INT", "PJ") or pd.isna(minute):
        return 0.0
    return float(minute) + (45 if isinstance(clock, str) and clock.startswith("+") else 0)


def build_match_data(frame: pd.DataFrame) -> list[dict]:
    frame = frame.assign(
        _period_rank=frame["period"].map(PERIOD_RANK).fillna(4),
        _sort_minute=[
            sort_minute(period, minute, clock)
            for period, minute, clock in zip(frame["period"], frame["minute"], frame["clock"])
        ],
    )
    frame = frame.sort_values(["game_id", "_period_rank", "_sort_minute"], kind="stable")
    matches = []
    for game_id, grp in frame.groupby("game_id", sort=True):
        first = grp.iloc[0]
        events = []
        for _, row in grp.iterrows():
            tags = [TAG_LABEL[t] for t in TAG_ORDER if row.get(t) is True]
            reason = row["reason_reported"] if pd.notna(row["reason_reported"]) else ""
            events.append([
                row["player"] if pd.notna(row["player"]) else "",
                canon_team(row["team"]),
                CARD_CODE[row["disciplinary_outcome"]],
                reason,
                tags,
                minute_label(row["period"], row["minute"], row["clock"]),
            ])
        matches.append({
            "id": str(game_id),
            "year": str(int(first["year"])),
            "date": str(first["match_date"]).strip(),
            "home": canon_team(first["home_team"]),
            "away": canon_team(first["away_team"]),
            "ref": first["referee"] if pd.notna(first["referee"]) else "",
            "events": events,
        })
    return matches


match_data = build_match_data(df)
len(match_data)

2543

## **4. `site-data`: agregados (manifest, tags, por time, por árbitro)**

`red_total`, `n_red` e `red` incluem expulsões por segundo amarelo e vermelhos diretos. Nos agregados por time e árbitro, `yellow` conta os registros da seção de cartões amarelos e `red` conta os registros da seção de cartões vermelhos. `total` é a soma dessas duas seções. `games` (partidas) vem de `games.csv` filtrado a `match_status == "played_with_report"`, contando todas as partidas do time/árbitro, tenha havido cartão ou não.

In [5]:
def build_site_data(frame: pd.DataFrame, games: pd.DataFrame) -> dict:
    total_events = len(frame)
    red_total = int(frame["disciplinary_outcome"].isin(["second_yellow", "direct_red"]).sum())

    tags = []
    for t in TAG_ORDER:
        sub = frame[frame[t] == True]  # noqa: E712
        n_eventos = len(sub)
        n_red = int(sub["disciplinary_outcome"].isin(["second_yellow", "direct_red"]).sum())
        tags.append({
            "tag": t,
            "label": TAG_LABEL[t],
            "n_textos": int(sub["reason"].nunique()),
            "n_eventos": int(n_eventos),
            "red_share": round(n_red / n_eventos, 4) if n_eventos else 0.0,
            "n_red": n_red,
        })
    tags.sort(key=lambda row: -row["n_eventos"])

    reported = games[games["match_status"] == "played_with_report"].copy()
    reported["home_c"] = reported["home_team"].map(canon_team)
    reported["away_c"] = reported["away_team"].map(canon_team)
    team_games: dict[str, int] = {}
    for _, r in reported.iterrows():
        team_games[r["home_c"]] = team_games.get(r["home_c"], 0) + 1
        team_games[r["away_c"]] = team_games.get(r["away_c"], 0) + 1
    ref_games = reported["referee"].value_counts().to_dict()

    frame = frame.copy()
    frame["team_c"] = frame["team"].map(canon_team)

    def aggregate(group_col: str, games_lookup: dict) -> dict:
        out = {}
        subset = frame[frame[group_col] != ""]
        for name, grp in subset.groupby(group_col):
            tag_counts = {t: int(grp[t].sum()) for t in TAG_ORDER if grp[t].sum() > 0}
            yellow_records = int((grp["disciplinary_outcome"] == "yellow").sum())
            second_yellows = int((grp["disciplinary_outcome"] == "second_yellow").sum())
            direct_reds = int((grp["disciplinary_outcome"] == "direct_red").sum())
            yellow = yellow_records
            red = second_yellows + direct_reds
            out[name] = {
                "total": yellow + red,
                "yellow": yellow,
                "red": red,
                "tags": tag_counts,
                "games": int(games_lookup.get(name, 0)),
            }
        return dict(sorted(out.items(), key=lambda kv: -kv[1]["total"]))

    return {
        "manifest": {
            "events": total_events,
            "red_total": red_total,
            "latest_match_date": var_manifest["latest_match_date"],
        },
        "tags": tags,
        "by_team": aggregate("team_c", team_games),
        "by_referee": aggregate("referee", ref_games),
        "team_colors": {
            name: {"primary": colors[0], "secondary": colors[1]}
            for name, colors in TEAM_COLORS.items()
        },
    }


site_data = build_site_data(df, games_df)
site_data["manifest"], len(site_data["by_team"]), len(site_data["by_referee"])

({'events': 14146, 'red_total': 928, 'latest_match_date': '2026-10-10'},
 30,
 79)

## **5. `var-data`: menções ao VAR por temporada, time, árbitro e partida**

Reúne as saídas do notebook `3`. Para os árbitros, o site mantém apenas quem possui mais de 10 partidas, o mesmo corte usado nos gráficos comparativos.

In [6]:
var_data = {
    "manifest": var_manifest,
    "by_year": [
        {"entity": str(int(row.entity)), "games": int(row.games), "var_games": int(row.var_games)}
        for row in var_season_df.itertuples(index=False)
    ],
    "by_team": [
        {"entity": str(row.entity), "games": int(row.games), "var_games": int(row.var_games)}
        for row in var_team_df.itertuples(index=False)
    ],
    "by_referee": [
        {"entity": str(row.entity), "games": int(row.games), "var_games": int(row.var_games)}
        for row in var_referee_df.loc[var_referee_df["games"].gt(10)].itertuples(index=False)
    ],
    "matches_with_var": var_matches_df.loc[
        var_matches_df["had_var"].eq(True), "game_id"
    ].astype(str).tolist(),
}

len(var_data["by_year"]), len(var_data["by_team"]), len(var_data["by_referee"])

(7, 30, 44)

## **6. `player-data`**

Já é gerado pelo notebook `2` em `data/processed/keyword_tags/site_player_data.json` (cartões, presenças e tags por jogador) — aqui só reaproveitamos o arquivo.

In [7]:
len(player_data), player_data[0]['name']

(2215, 'Kannemann')

## **7. Gravar no `index.html` da página**

Substitui só o conteúdo dos quatro blocos `<script id="..." type="application/json">`, mantendo todo o resto do arquivo (CSS, fontes, JS de renderização) intacto.

In [8]:
def replace_block(html: str, block_id: str, payload) -> str:
    pattern = re.compile(
        r'(<script id="' + re.escape(block_id) + r'" type="application/json">)(.*?)(</script>)',
        re.DOTALL,
    )
    new_json = json.dumps(payload, ensure_ascii=False, separators=(",", ":"))
    new_html, n = pattern.subn(lambda m: m.group(1) + new_json + m.group(3), html, count=1)
    assert n == 1, f"bloco {block_id!r} não encontrado em {PAGE_HTML}"
    return new_html


html = PAGE_HTML.read_text(encoding="utf-8")
html = replace_block(html, "match-data", match_data)
html = replace_block(html, "site-data", site_data)
html = replace_block(html, "player-data", player_data)
html = replace_block(html, "var-data", var_data)
PAGE_HTML.write_text(html, encoding="utf-8")

print(f"escrito em {PAGE_HTML}")
print(f"partidas: {len(match_data)} | eventos: {site_data['manifest']['events']} | "
      f"cartões vermelhos: {site_data['manifest']['red_total']} | "
      f"times: {len(site_data['by_team'])} | árbitros: {len(site_data['by_referee'])} | "
      f"jogadores: {len(player_data)} | partidas VAR: {len(var_data['matches_with_var'])}")

escrito em /Users/gabrielcarbinatto/Documents/GitHub/cbf-match-report-analysis/index.html
partidas: 2543 | eventos: 14146 | cartões vermelhos: 928 | times: 30 | árbitros: 79 | jogadores: 2215 | partidas VAR: 1131


## **8. Próximo passo**

Este notebook não faz commit nem push. Na raiz deste repositório, confira o `git diff` e publique quando estiver satisfeito:

```bash
git diff --stat
git add index.html
git commit -m "Atualiza dados do site"
git push
```